# Optimización multiobjetivo de arquitecturas MLP para IA eficiente
### NSGA-II frente a búsqueda aleatoria con el mismo presupuesto de evaluaciones

> **Cómo leer este cuaderno.** Cada afirmación empírica proviene de una celda ejecutable; los números del texto se
> imprimen desde los CSV de `resultados/`. Se distingue siempre entre **resultado** (lo medido), **interpretación**
> (lo que creemos que significa) e **hipótesis** (lo que aún no se ha comprobado).

## 2. Resumen
Se evaluó si NSGA-II, implementado desde cero, encuentra mejores compromisos entre **error de validación** y **número de parámetros** que una **búsqueda aleatoria con el mismo presupuesto** (200 redes entrenadas por método) en perceptrones multicapa pequeños sobre el conjunto *Digits* (1797 muestras, 64 variables, 10 clases).
Se hicieron 10 repeticiones pareadas (misma semilla y partición 60/20/20), sin usar el conjunto de prueba durante la optimización, y se compararon con el hipervolumen normalizado de lo evaluado, con una referencia común, y con la prueba de Wilcoxon pareada exacta.

**Resultado principal.** NSGA-II tuvo mayor hipervolumen en 8 de 10 pares (mediana 0,9198 frente a 0,9111), pero $p=0{,}106$: la hipótesis **no queda confirmada**; es una tendencia favorable, no concluyente. Con parámetros en escala lineal no hay ventaja (5 a 5). El compromiso error–tamaño sí es real: reducir los parámetros de ≈ 2400 a ≈ 1250 costó ≈ 0,8 puntos porcentuales de error de prueba.
**Alcance:** un solo conjunto de datos fácil, 10 semillas y entrenamiento corto (50–54 % de los entrenamientos no convergieron en 100 épocas). El número de parámetros es una medida estructural: **no se midió energía ni latencia**.
**Palabras clave:** optimización multiobjetivo, NSGA-II, frente de Pareto, redes neuronales eficientes, búsqueda aleatoria, hipervolumen.

## 3. Introducción
Los modelos de aprendizaje automático suelen compararse por su desempeño predictivo, pero desplegarlos exige también
recursos: memoria, tiempo de cómputo y energía. Una red neuronal más grande no siempre es mejor: a partir de cierto
tamaño la mejora es marginal y el costo sigue creciendo. Elegir la arquitectura es, por tanto, un problema con
**objetivos en conflicto**, y la computación evolutiva multiobjetivo es una herramienta natural para estudiarlo porque
devuelve *un conjunto* de compromisos (el frente de Pareto) en lugar de una única "mejor" solución.

## 4. Planteamiento del problema
Dado un conjunto de datos de clasificación y un espacio de arquitecturas MLP pequeñas, ¿cómo encontrar, con un número
limitado de entrenamientos, arquitecturas que ofrezcan buenos compromisos entre **error de validación** y **número de
parámetros entrenables**? Cada evaluación implica entrenar una red, de modo que el presupuesto de evaluaciones es el
recurso escaso que debe igualarse al comparar métodos.

## 5. Justificación
* Conecta **computación evolutiva** (NSGA-II) con **redes neuronales** (MLP), dos pilares de la computación blanda.
* El resultado es útil para decidir arquitecturas en dispositivos con memoria limitada.
* Aporta un protocolo de comparación justa (mismo presupuesto, mismas particiones, diseño pareado) reproducible en un
  computador convencional.

## 6. Pregunta de investigación
**¿Qué tan eficaz es NSGA-II para encontrar arquitecturas de redes neuronales pequeñas que equilibren el error predictivo
y el número de parámetros, en comparación con una búsqueda aleatoria que usa el mismo presupuesto de evaluaciones?**

## 7. Hipótesis
**H1.** NSGA-II encuentra un conjunto de arquitecturas no dominadas que representa mejor el compromiso error–complejidad
que una búsqueda aleatoria con el mismo presupuesto.
*Operacionalización:* mayor hipervolumen normalizado (referencia común) del archivo de arquitecturas evaluadas, comparado
con la prueba de Wilcoxon de rangos con signo sobre pares con la misma semilla. **H0:** no hay diferencia sistemática.
La hipótesis **no se asume verdadera**: puede rechazarse y el trabajo seguiría siendo válido.

## 8. Objetivos
**General.** Evaluar experimentalmente si NSGA-II supera a la búsqueda aleatoria, con igual presupuesto, para hallar
compromisos error–parámetros en MLP pequeñas.

**Específicos.**
1. Formular la búsqueda de arquitecturas como un problema de dos objetivos con una codificación válida por construcción.
2. Implementar y verificar NSGA-II desde cero (ordenamiento no dominado, crowding, torneo, cruce, mutación, elitismo).
3. Implementar una búsqueda aleatoria con el mismo espacio, muestreador y presupuesto.
4. Ejecutar 10 repeticiones pareadas por método, sin usar el conjunto de prueba durante la optimización.
5. Comparar con hipervolumen, tamaño de frente, mejor error y parámetros mínimos, con dispersión y prueba estadística.
6. Realizar un análisis de sensibilidad acotado (tamaño de población y presupuesto) y documentar las limitaciones.

## 9. Fundamentos teóricos

### 9.1 Inteligencia artificial eficiente
*Eficiencia* significa obtener un desempeño dado consumiendo menos recursos. Conviene distinguir tres clases, porque **no son equivalentes**:

| Tipo | Qué mide | Ejemplos de indicador |
|---|---|---|
| Estructural | Tamaño del modelo | nº de parámetros, memoria |
| Temporal | Tiempo de cómputo | latencia de inferencia, tiempo de entrenamiento |
| Energética | Energía consumida | julios por inferencia (requiere un medidor) |

Reducir el tamaño importa en dispositivos con memoria, batería o cómputo limitados (sensores, móviles, microcontroladores).
**Advertencia que respetamos en todo el trabajo:** el número de parámetros es una medida *estructural*; no es una medición de
tiempo ni de energía, y aquí **no se mide energía**, por lo que no afirmaremos ahorro energético.

### 9.2 Optimización multiobjetivo
En un problema **monoobjetivo** se minimiza una sola función $f(x)$ y existe (en general) un óptimo. En uno **multiobjetivo**
se minimiza un vector $\mathbf f(x)=(f_1(x),\dots,f_m(x))$ y los objetivos suelen estar en conflicto, así que no hay una
solución que sea mejor en todo. Elementos: *espacio de soluciones* $\mathcal X$ (aquí, las arquitecturas permitidas),
*funciones objetivo* ($f_1$ = error de validación, $f_2$ = parámetros) y *restricciones* (aquí, límites de capas y neuronas).

**Dominancia de Pareto (minimización).** $a$ domina a $b$, escrito $a\prec b$, si $f_i(a)\le f_i(b)$ para todo $i$ y
$f_j(a)<f_j(b)$ para al menos un $j$. Una solución es **no dominada** si ninguna otra la domina; el conjunto de las no dominadas es
el **frente de Pareto** (de lo evaluado, o el verdadero si se conoce el óptimo global).

**Ejemplo numérico.** Sean A = (error 0,04; 600 parámetros), B = (0,05; 900), C = (0,03; 1500), D = (0,04; 600):
* A ≺ B: tiene menos error **y** menos parámetros → B está *dominada*.
* A y C **no se dominan**: C tiene menos error pero más parámetros → ambas son *no dominadas* (representan compromisos distintos).
* A y D son iguales: no hay dominancia entre ellas.

**Elegir una solución final.** El frente ofrece opciones, no una respuesta; la elección depende de una preferencia externa. En este trabajo
se usan tres criterios definidos *a priori* y sobre **validación**: (i) *menor error*, (ii) *codo* (la más cercana al ideal tras normalizar
ambos objetivos) y (iii) *compacta* (la de menos parámetros con error ≤ mejor + 1 punto porcentual), que corresponde al caso de un dispositivo limitado.

### 9.3 Redes neuronales MLP
Una **MLP** (perceptrón multicapa) es una red de capas densas: entrada → capas ocultas → salida. Cada neurona calcula
$a=\phi(\mathbf w^\top\mathbf x+b)$ con **pesos** $\mathbf w$, **sesgo** $b$ y **función de activación** $\phi$ (ReLU $=\max(0,z)$; tanh).
Se entrena minimizando una pérdida (entropía cruzada) con gradiente (Adam); se **valida** en datos no usados para ajustar pesos.

**Parámetros entrenables.** Con $n_0$ entradas, capas ocultas $n_1,\dots,n_L$ y $n_{L+1}$ salidas:
$$P=\sum_{l=1}^{L+1}(n_{l-1}+1)\,n_l .$$
Ejemplo (Digits: $n_0=64$, 10 clases) con una capa de 16 neuronas: $(64+1)\cdot16+(16+1)\cdot10=1040+170=\mathbf{1210}$.
Con 8 y 4 neuronas: $65\cdot8+9\cdot4+5\cdot10=520+36+50=\mathbf{606}$. (Estas cifras se verifican con pruebas automáticas contra scikit-learn.)
La arquitectura determina la complejidad: los parámetros crecen aproximadamente con el producto de neuronas de capas consecutivas.

### 9.4 Algoritmo NSGA-II
**Origen.** Propuesto por Deb, Pratap, Agarwal y Meyarivan (2002, *IEEE Trans. Evol. Comput.* 6(2), 182–197) para corregir tres críticas a NSGA:
complejidad $O(MN^3)$, falta de elitismo y necesidad de un parámetro de compartición. NSGA-II ofrece ordenamiento no dominado rápido $O(MN^2)$,
elitismo por combinación padres+hijos y la **distancia de crowding** para preservar diversidad sin parámetros.

**Funcionamiento paso a paso**
1. *Inicialización:* población $P_0$ de $N$ individuos válidos; se evalúan los objetivos (aquí, se entrena cada red).
2. *Ordenamiento no dominado:* se parte la población en frentes $F_0,F_1,\dots$ ($F_0$ = no dominadas).
3. *Crowding:* dentro de cada frente, a cada individuo se le asigna una medida de lo "aislado" que está; los extremos reciben $\infty$.
4. *Selección por torneo binario:* se eligen 2 individuos al azar; gana el de menor rango y, si empatan, el de mayor crowding.
5. *Cruce y mutación:* los padres seleccionados producen hijos $Q_t$ ($N$ hijos).
6. *Reemplazo elitista:* se unen $R_t=P_t\cup Q_t$ ($2N$), se ordena y se llenan $N$ plazas frente por frente; el último frente que no cabe se corta por mayor crowding.
7. *Parada:* tras $G$ generaciones (aquí, hasta agotar el presupuesto de evaluaciones).

**Pseudocódigo**
```
P0 <- N individuos aleatorios válidos ; evaluar(P0)                  # N evaluaciones
for t = 0 .. G-1:
    rango, crowding <- ordenar_no_dominado(P_t), crowding(P_t)
    Q_t <- {}
    while |Q_t| < N:
        p1, p2 <- torneo_binario(P_t), torneo_binario(P_t)
        h <- mutar(cruzar(p1, p2))     # se reintenta si el hijo ya fue evaluado
        Q_t <- Q_t ∪ {h}
    evaluar(Q_t)                                                       # N evaluaciones
    R_t <- P_t ∪ Q_t
    P_{t+1} <- primeros N de R_t por (rango asc., crowding desc.)
devolver el frente no dominado de lo evaluado
```
**Contabilidad:** evaluaciones $=N+G\cdot N$. *Iteraciones* (generaciones) ≠ *evaluaciones*; el presupuesto compartido se cuenta en evaluaciones.

**Distancia de crowding.** Para cada objetivo $k$, se ordena el frente; los extremos reciben $\infty$ y cada interior suma
$\dfrac{f_k(i+1)-f_k(i-1)}{f_k^{\max}-f_k^{\min}}$. Se prefiere mayor distancia (zona menos poblada).

**Ventajas:** simple, elitista, sin parámetro de compartición, buen esparcimiento. **Limitaciones:** el crowding pierde eficacia con muchos objetivos
(≥4); el rendimiento depende de los operadores y de la codificación; con presupuestos pequeños puede no superar a una búsqueda aleatoria.

In [1]:
# Ejemplo numérico de NSGA-II: reemplazo elitista con 2N = 8 candidatos y N = 4 plazas.
import numpy as np
from src.pareto import ordenamiento_no_dominado, distancia_crowding, seleccionar_supervivientes

# columnas: (error de validación, parámetros). Primeras 4 = padres; últimas 4 = hijos.
F = np.array([[0.10, 300], [0.05, 900], [0.03, 2000], [0.20, 200],
              [0.06, 700], [0.04, 1200], [0.12, 350], [0.03, 2500]], float)
frentes = ordenamiento_no_dominado(F)
print("Frentes (índices):", frentes)
elegidos, rango, crowd = seleccionar_supervivientes(F, 4)
for i in range(len(F)):
    print(f"  ind {i}: err={F[i,0]:.2f} params={F[i,1]:>5.0f} rango={rango[i]} crowding={crowd[i]:.2f} "
          f"-> {'SOBREVIVE' if i in set(elegidos) else 'descartado'}")
print("El frente 0 completo cabe en las 4 plazas: se conservan compromisos distintos (barato/impreciso ... caro/preciso).")

Frentes (índices): [[0, 1, 2, 3, 4, 5], [6, 7]]
  ind 0: err=0.10 params=  300 rango=0 crowding=1.10 -> SOBREVIVE
  ind 1: err=0.05 params=  900 rango=0 crowding=0.40 -> descartado
  ind 2: err=0.03 params= 2000 rango=0 crowding=inf -> SOBREVIVE
  ind 3: err=0.20 params=  200 rango=0 crowding=inf -> SOBREVIVE
  ind 4: err=0.06 params=  700 rango=0 crowding=0.63 -> descartado
  ind 5: err=0.04 params= 1200 rango=0 crowding=0.73 -> SOBREVIVE
  ind 6: err=0.12 params=  350 rango=1 crowding=inf -> descartado
  ind 7: err=0.03 params= 2500 rango=1 crowding=inf -> descartado
El frente 0 completo cabe en las 4 plazas: se conservan compromisos distintos (barato/impreciso ... caro/preciso).


### 9.5 Búsqueda aleatoria
Consiste en muestrear configuraciones al azar del mismo espacio, evaluarlas y quedarse con las mejores (aquí, con el frente no dominado de lo evaluado).
Bergstra y Bengio (2012) la proponen como **línea base natural** para juzgar métodos adaptativos: es simple, no tiene sesgo de diseño y su rendimiento depende solo del presupuesto.
**Comparación justa:** mismo espacio de búsqueda, **mismo número de arquitecturas entrenadas**, mismo procedimiento de entrenamiento, mismas particiones, mismas semillas, y mismo
criterio de construcción del frente (aquí, el *archivo* de todo lo evaluado para ambos métodos).

## 10. Estado del arte y matriz bibliográfica
Los datos bibliográficos de la tabla fueron **comprobados en las páginas de la editorial, de arXiv o de índices bibliográficos (dblp) el 15-nov-2026**.
El estado de publicación se indica explícitamente: un *preprint* no se presenta como artículo arbitrado.

| ID | Referencia y enlace | Año / estado | Problema y método | Datos y comparación | Hallazgo y limitación | Relación con nuestra pregunta |
|---|---|---|---|---|---|---|
| F1 | Deb, Pratap, Agarwal, Meyarivan. *A Fast and Elitist Multiobjective Genetic Algorithm: NSGA-II.* IEEE TEVC 6(2):182–197. DOI [10.1109/4235.996017](https://doi.org/10.1109/4235.996017) | 2002 · artículo arbitrado (clásico) | Corrige costo, elitismo y compartición de NSGA; ordenamiento rápido + crowding | Problemas de prueba; compara con PAES y SPEA | Mejor esparcimiento y convergencia en la mayoría de problemas de prueba; evaluado en funciones sintéticas, no en entrenamiento de redes | **Método principal** que implementamos |
| F2 | Zitzler, Thiele. *Multiobjective evolutionary algorithms: a comparative case study and the strength Pareto approach.* IEEE TEVC 3(4):257–271. DOI [10.1109/4235.797969](https://doi.org/10.1109/4235.797969) | 1999 · artículo arbitrado (clásico) | Compara MOEAs cuantitativamente y propone SPEA; formaliza la comparación de conjuntos no dominados | Problema de la mochila 0/1 extendido | Compara cuantitativamente cuatro MOEAs y propone SPEA; su caso es una mochila, no redes | Base de las métricas de comparación de conjuntos no dominados que usamos (espacio cubierto/hipervolumen y cobertura C) |
| F3 | Bergstra, Bengio. *Random Search for Hyper-Parameter Optimization.* JMLR 13(10):281–305. [jmlr.org](https://jmlr.org/papers/v13/bergstra12a.html) | 2012 · artículo arbitrado | Búsqueda aleatoria vs rejilla/manual para configurar redes | Redes y DBN en varios conjuntos | Aleatoria como línea base natural; la mayoría de hiperparámetros importa poco; es monoobjetivo | Justifica la **referencia** y su configuración |
| F4 | López-González, Rodríguez, Greiner. *A Multi-Objective Evolutionary Computation Approach for Improving Neural Network-Based Surrogate Models in Structural Engineering.* Algorithms 18(12):754. DOI [10.3390/a18120754](https://doi.org/10.3390/a18120754) | 2025 · artículo arbitrado (publicado 28-nov-2025) | NAS multiobjetivo con NSGA-II (precisión vs tiempo de entrenamiento) en sustitutos de pórticos de acero | Compara configuraciones de NSGA-II con búsqueda aleatoria; significancia sobre el hipervolumen (según su resumen) | Muestra utilidad de NSGA-II en NAS; dominio y objetivos distintos a los nuestros (tiempo, no parámetros; regresión) | **Antecedente más cercano**: mismo par de métodos y misma métrica |
| F5 | Vo, Luong. *Efficient Multi-Objective Neural Architecture Search via Pareto Dominance-based Novelty Search.* GECCO 2024. arXiv [2407.20656](https://arxiv.org/abs/2407.20656), DOI de actas [10.1145/3638529.3654064](https://doi.org/10.1145/3638529.3654064) | 2024 · aceptado como artículo completo en GECCO 2024 (según arXiv) | NAS multiobjetivo con métricas sin entrenamiento y novedad basada en dominancia | No verificado en detalle (solo se leyó el resumen); compara contra NAS multiobjetivo convencional según el resumen | Señala la convergencia prematura y pérdida de diversidad de métodos guiados por objetivos; usa métricas proxy, no entrenamiento | Contexto de **diversidad** y de los riesgos de convergencia en NAS multiobjetivo |
| F6 | Khaled, Kaymak, Genga. *Alternating Bi-Objective Optimization for Explainable Neuro-Fuzzy Systems.* arXiv [2602.19253](https://arxiv.org/abs/2602.19253) | 2026 · **preprint** (el documento del docente indica IEEE CAI 2026; no lo confirmé en la ficha de arXiv) | Compromiso exactitud–explicabilidad en ANFIS con optimización bi-objetivo por gradiente | ≈5000 experimentos en 9 conjuntos de regresión de UCI | Recupera soluciones fuera del casco convexo del frente evolutivo; objetivo de explicabilidad ≠ parámetros; sistemas neurodifusos | Muestra que **el compromiso se estudia con frentes** y que los métodos evolutivos son costosos |
| F7 | ACM SIGEVO. *GECCO 2026 — Neuroevolution track.* [gecco-2026.sigevo.org](https://gecco-2026.sigevo.org/Track?itemId=59) | 2026 · descripción oficial (contexto, no artículo) | Alcance del track: arquitecturas, métodos multiobjetivo, eficiencia | — | Delimita la línea de investigación vigente | Contextualiza la actualidad del tema |

**Nivel de lectura (honestidad bibliográfica).** F1, F3 y F4: resumen y partes del texto completo consultadas en línea; F2, F5, F6, F7: resumen e introducción. No se reprodujeron resultados publicados ni se compararon números con los nuestros, que usan otros datos y objetivos.

**Síntesis crítica.** Los trabajos coinciden en tres ideas: (1) el compromiso rendimiento–costo se estudia con **frentes de Pareto** y no con un único número (F1, F4, F6);
(2) la **búsqueda aleatoria** es una referencia razonable y exigente (F3, F4); y (3) los métodos guiados por objetivos pueden perder diversidad o converger prematuramente (F5). Difieren en el **objetivo de costo**
(tiempo de entrenamiento en F4, explicabilidad en F6, métricas proxy en F5, parámetros en nuestro caso), en el tipo de modelo y en la evidencia (F1 y F2 usan problemas sintéticos).
Por eso no podemos generalizar a nuestros datos las afirmaciones de superioridad de NSGA-II: **la pregunta pequeña** que sí podemos examinar es si, en un espacio MLP acotado, con 200 entrenamientos y
sobre Digits, NSGA-II mejora el hipervolumen frente a la búsqueda aleatoria.

## 11. Diseño metodológico

| Elemento | Decisión | Justificación |
|---|---|---|
| Método principal | NSGA-II implementado desde cero | Permite explicar y verificar cada operador; evita una dependencia opaca |
| Referencia | Búsqueda aleatoria (mismo espacio, muestreador, presupuesto) | Línea base estándar (Bergstra y Bengio, 2012) |
| Objetivo 1 | $f_1$ = error de clasificación en **validación** $=1-\text{accuracy}_{val}$ | Interpretable; Digits está balanceado (~10 % por clase), por lo que accuracy es razonable |
| Objetivo 2 | $f_2$ = número de parámetros entrenables $P$ (fórmula verificada) | Medida estructural exacta y determinista |
| Presupuesto | **200 evaluaciones reales** por método y corrida (= 200 redes entrenadas) | Se cuentan entrenamientos, no iteraciones; el evaluador lo impone y lo registra |
| NSGA-II | $N=20$, $G=9$ generaciones, cruce uniforme ($p=0{,}9$), mutación por gen ($p=1/5$), torneo binario por crowding | $20+9\cdot20=200$ |
| Repeticiones | **10 semillas** (0–9), diseño pareado | Mínimo sugerido por el docente |
| Particiones | Estratificadas 60/20/20, **una por semilla**, idénticas para ambos métodos | Controla la variabilidad de datos; prueba solo al final |
| Entrenamiento | `MLPClassifier`, Adam, lr = 0,01, 100 épocas, semilla de pesos = semilla de la corrida | Idéntico para toda arquitectura y método |
| Duplicados | Ningún método reevalúa una arquitectura ya evaluada | El entrenamiento es determinista: repetir desperdicia presupuesto |
| Pareado | La población inicial de NSGA-II = las primeras 20 muestras de la aleatoria | Reduce varianza; las diferencias nacen de cómo se gastan las 180 restantes |
| Métrica principal | Hipervolumen normalizado del **archivo** de arquitecturas evaluadas, referencia común $(0{,}9;\ P_{\max})$ | Resume convergencia y diversidad en un número; ver sección 19 |
| Métricas secundarias | Tamaño del frente, mejor error, parámetros mínimos con error ≤ mejor + 1 pp, tiempo, error de prueba de las soluciones elegidas | Complementan; el tiempo es solo informativo |
| Prueba estadística | Wilcoxon de rangos con signo (exacto, bilateral) sobre los 10 pares | No exige normalidad; tamaño de muestra pequeño |

**Referencias de contexto (fuera del presupuesto):** tres MLP manuales (16; 64; 64-64-64 neuronas, ReLU), entrenadas con el mismo procedimiento, para situar los resultados.

**Riesgo principal y alternativa:** que NSGA-II no supere a la aleatoria con 200 evaluaciones. Se reportaría igualmente (resultado negativo válido); la alternativa de menor alcance es el análisis del frente de una sola corrida.

## 12. Descripción del conjunto de datos
* **Nombre:** *Optical Recognition of Handwritten Digits* (copia del *test set* incluida en scikit-learn: `sklearn.datasets.load_digits`).
* **Origen:** E. Alpaydin y C. Kaynak (1998), Universidad de Boğaziçi; UCI Machine Learning Repository, DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49).
* **Licencia:** CC BY 4.0 (ficha de UCI). Se reconoce la autoría.
* **Contenido:** 1797 imágenes de 8×8 píxeles (enteros 0–16) → **64 variables**, 10 clases (dígitos 0–9), sin valores faltantes.
  La ficha de UCI describe 5620 instancias en total (3823 de entrenamiento y 1797 de prueba, de 43 personas); scikit-learn incluye únicamente las 1797 de prueba. Aquí **se re-particiona** ese conjunto 60/20/20 de forma estratificada.
* **Por qué Digits y no Breast Cancer.** En un piloto con 40 arquitecturas aleatorias (una partición, una semilla), Breast Cancer dio errores de validación en un rango estrecho (2,6 %–6,1 %; con ~114 muestras de validación, cada muestra equivale a ~0,9 puntos) y un frente de solo 4 arquitecturas distintas: casi todas las redes rinden parecido y el compromiso es poco visible.
  Digits dio errores de 1,9 % a 52,9 % y un frente de 8 arquitecturas distintas, de modo que el compromiso es observable. *(Es una decisión de diseño basada en un piloto, no un resultado de la investigación.)*
* **Limitación:** Digits es un problema fácil y de baja resolución; las conclusiones no se generalizan a datos más complejos.

## 13. Arquitectura experimental
Estructura del proyecto (todo con rutas relativas):
```
src/            config · datos · arquitectura · evaluacion · pareto · nsga2 · busqueda_aleatoria · experimento · metricas · graficos
tests/          pruebas automáticas (fórmulas, NSGA-II vs fuerza bruta, hipervolumen vs malla, presupuesto, equidad)
resultados/     CSV de cada corrida y tablas finales        figuras/   gráficos generados por este cuaderno
ejecutar_experimentos.py   reproduce todas las corridas (reanudable)
```
**Flujo:** semilla → partición 60/20/20 → (NSGA-II | aleatoria) con un `EvaluadorPresupuestado` que cuenta entrenamientos → archivo de arquitecturas evaluadas → frente no dominado →
métricas → selección final con validación → **única** medición en prueba.

In [2]:
# --- Configuración global y semillas ---------------------------------------------------------
import sys, json, inspect, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

sys.path.insert(0, str(Path.cwd()))          # el cuaderno vive en la raíz del proyecto
if not (Path.cwd() / "src").is_dir():
    raise FileNotFoundError("Ejecute el cuaderno desde la carpeta del proyecto (la que contiene src/). En Colab: %cd /content/nsga2_mlp_computacion_blanda")
from src.config import (EspacioBusqueda, ConfigEntrenamiento, ConfigBusqueda, ConfigDatos,
                        ConfigMetricas, SEMILLAS_POR_DEFECTO)
from src.datos import particionar
from src.arquitectura import Arquitectura, contar_parametros
from src.experimento import (ejecutar_corrida, cargar_resultados, evaluar_referencias_manuales)
from src import metricas as M, graficos as G, nsga2, busqueda_aleatoria, pareto, arquitectura as A, evaluacion

pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30)
MODO = "guardados"     # "guardados": lee resultados/ · "completo": vuelve a ejecutar todo (~30 min en 1 CPU)
DIR_RES, DIR_FIG = Path("resultados"), Path("figuras")
SEMILLAS = list(SEMILLAS_POR_DEFECTO)
if MODO == "completo":
    subprocess.run([sys.executable, "ejecutar_experimentos.py", "--modo", "completo", "--rehacer"], check=True)
if not (DIR_RES / "corridas").exists():
    raise FileNotFoundError("Faltan resultados/: ejecute `python ejecutar_experimentos.py` o ponga MODO='completo'.")
print("Espacio:", EspacioBusqueda()); print("Entrenamiento:", ConfigEntrenamiento())
print("Búsqueda (principal):", ConfigBusqueda()); print("Métricas:", ConfigMetricas())
print("Entorno usado en las corridas:", json.loads((DIR_RES / "entorno.json").read_text())["scikit-learn"], "(scikit-learn)")

Espacio: EspacioBusqueda(capas_min=1, capas_max=3, unidades_min=2, unidades_max=64, activaciones=('relu', 'tanh'))
Entrenamiento: ConfigEntrenamiento(max_iter=100, tasa_aprendizaje=0.01, solver='adam')
Búsqueda (principal): ConfigBusqueda(presupuesto=200, tam_poblacion=20, prob_cruce=0.9, sigma_mutacion=0.6, max_intentos_hijo=100)
Métricas: ConfigMetricas(error_ref=0.9, tolerancia_compacta=0.01)
Entorno usado en las corridas: 1.9.1 (scikit-learn)


In [3]:
# --- Datos: tamaños de partición y comprobaciones anti-fuga ------------------------------------
d = particionar(0)
print(f"Entrenamiento {d.X_ent.shape}, validación {d.X_val.shape}, prueba {d.X_prueba.shape}; clases = {d.n_clases}")
print("Media del escalador en entrenamiento ≈ 0 :", bool(np.allclose(d.X_ent.mean(0), 0, atol=1e-8)),
      "| en validación ≈ 0 :", bool(np.allclose(d.X_val.mean(0), 0, atol=1e-8)), "(debe ser False: solo se ajusta con entrenamiento)")
print("Proporción por clase en entrenamiento:", np.round(np.bincount(d.y_ent) / len(d.y_ent), 3))
pmin, pmax = M.extremos_parametros()
print(f"Parámetros de la red más pequeña del espacio: {pmin} · de la más grande: {pmax}")
tamano_espacio = sum((EspacioBusqueda().unidades_max - EspacioBusqueda().unidades_min + 1) ** L for L in (1, 2, 3)) * 2
print(f"Tamaño del espacio de arquitecturas (fenotipos): {tamano_espacio:,}")

Entrenamiento (1078, 64), validación (359, 64), prueba (360, 64); clases = 10
Media del escalador en entrenamiento ≈ 0 : True | en validación ≈ 0 : False (debe ser False: solo se ajusta con entrenamiento)
Proporción por clase en entrenamiento: [0.099 0.101 0.098 0.102 0.101 0.101 0.101 0.099 0.096 0.1  ]
Parámetros de la red más pequeña del espacio: 160 · de la más grande: 13130
Tamaño del espacio de arquitecturas (fenotipos): 508,158


## 14. Implementación de NSGA-II
Se implementó **desde cero** (sin DEAP) para poder explicar cada operador. Decisiones principales:
* **Codificación:** genoma `(n_capas, u1, u2, u3, activación)`; solo las primeras `n_capas` unidades forman el fenotipo. Decodificar valida rangos; cruce y mutación recortan al rango, por lo que **no generan arquitecturas inválidas**.
* **Cruce:** uniforme (cada gen de un padre al azar). **Mutación:** por gen con $p=1/5$; capas ±1; unidades × $e^{\mathcal N(0,0{,}6)}$; activación alternada.
* **Sin duplicados:** si un hijo ya fue evaluado se reintenta (hasta 100 veces); si fallara se usa una muestra aleatoria única y se **contabiliza** (`hijos_fallback_aleatorio`; ver sección 17).
* **Presupuesto:** `validar_presupuesto_nsga2` exige que $(B-N)$ sea múltiplo de $N$; el `EvaluadorPresupuestado` lanza error si se intenta evaluar de más.

Código clave (se muestra el que realmente se ejecutó):

In [4]:
for f in (pareto.ordenamiento_no_dominado, pareto.distancia_crowding, pareto.seleccionar_supervivientes):
    print(inspect.getsource(f))

def ordenamiento_no_dominado(F):
    """Fast non-dominated sort (Deb et al., 2002). Devuelve lista de frentes (índices).

    Complejidad O(M N^2): para cada p cuenta cuántas soluciones lo dominan (n_p) y
    guarda las que él domina (S_p). El frente 0 son las de n_p = 0; al retirarlas se
    decrementa n_q de las dominadas y emerge el siguiente frente.
    """
    F = _como_matriz(F)
    n = len(F)
    dominadas_por = [[] for _ in range(n)]   # S_p
    n_dominan = np.zeros(n, dtype=int)       # n_p
    for p in range(n):
        for q in range(p + 1, n):
            if domina(F[p], F[q]):
                dominadas_por[p].append(q)
                n_dominan[q] += 1
            elif domina(F[q], F[p]):
                dominadas_por[q].append(p)
                n_dominan[p] += 1
    frentes = [[i for i in range(n) if n_dominan[i] == 0]]
    while frentes[-1]:
        siguiente = []
        for p in frentes[-1]:
            for q in dominadas_por[p]:
                n_dominan[q] -= 1
    

In [5]:
for f in (nsga2._torneo_binario, nsga2._generar_hijo, nsga2.ejecutar_nsga2):
    print(inspect.getsource(f))

def _torneo_binario(rango, crowd, rng):
    """Comparación por crowding: menor rango gana; si empatan, mayor crowding."""
    i, j = rng.choice(len(rango), size=2, replace=False)
    if rango[i] != rango[j]:
        return int(i if rango[i] < rango[j] else j)
    if crowd[i] != crowd[j]:
        return int(i if crowd[i] > crowd[j] else j)
    return int(rng.choice([i, j]))

def _generar_hijo(genomas, rango, crowd, rng, espacio, cfg, vistos):
    """Torneo -> cruce -> mutación, reintentando hasta un fenotipo NUEVO.

    Devuelve (genoma, fue_fallback). Si tras max_intentos_hijo no se logra, se usa una
    muestra aleatoria única (se contabiliza para reportarlo, no se oculta).
    """
    for _ in range(cfg.max_intentos_hijo):
        p1 = genomas[_torneo_binario(rango, crowd, rng)]
        p2 = genomas[_torneo_binario(rango, crowd, rng)]
        hijo = cruce_uniforme(p1, p2, rng) if rng.random() < cfg.prob_cruce else p1
        hijo = mutar(hijo, rng, espacio, cfg.sigma_mutacion)
      

**Verificación.** Las piezas matemáticas se comprobaron contra implementaciones independientes: ordenamiento no dominado vs fuerza bruta (20 conjuntos aleatorios con empates), hipervolumen vs integración por malla fina (20 conjuntos),
crowding y supervivientes con ejemplos calculados a mano, fórmula de parámetros contra los tensores de scikit-learn, y propiedades de presupuesto, duplicados y reproducibilidad. La siguiente celda las ejecuta:

In [6]:
import importlib.util
if importlib.util.find_spec("pytest") is None:
    print("pytest no está instalado: omita esta celda o ejecute `pip install pytest` (las pruebas están en tests/).")
else:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider"], capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1])
    assert r.returncode == 0, r.stdout[-2000:]

62 passed in 27.09s


## 15. Implementación de búsqueda aleatoria
Usa el **mismo** espacio, el **mismo** muestreador (logarítmico en neuronas) y el **mismo** evaluador con presupuesto. Comparte el flujo aleatorio con la población inicial de NSGA-II,
de modo que sus primeras $N=20$ arquitecturas coinciden (diseño pareado).

In [7]:
print(inspect.getsource(busqueda_aleatoria.ejecutar_busqueda_aleatoria))

def ejecutar_busqueda_aleatoria(evaluador, espacio=EspacioBusqueda(), semilla=0):
    """Evalúa `presupuesto` arquitecturas distintas muestreadas al azar.

    Usa el mismo generador (semilla, 0) que la población inicial de NSGA-II, de modo
    que sus primeras N evaluaciones coinciden (diseño pareado): las diferencias
    posteriores se deben únicamente a cómo se invierte el resto del presupuesto.
    """
    if evaluador.n_evaluaciones != 0:
        raise ValueError("El evaluador debe estar vacío al iniciar")
    t0 = time.perf_counter()
    rng_init = np.random.default_rng([semilla, 0])
    for _ in range(evaluador.presupuesto):
        g = muestrear_genoma_unico(rng_init, espacio, evaluador.vistos)
        evaluador.evaluar(decodificar(g, espacio), generacion=0)
    if evaluador.n_evaluaciones != evaluador.presupuesto:
        raise AssertionError("La búsqueda aleatoria no consumió exactamente el presupuesto")
    return ResultadoBusqueda("Aleatoria", semilla, evaluador.registro, t

## 16. Ejecución de experimentos
**Protocolo ejecutado** (`ejecutar_experimentos.py`): 10 semillas (0–9) × 2 métodos × 200 evaluaciones (conjunto *principal*), más NSGA-II con $N=10$ y $N=50$ (sensibilidad, 10 semillas, 200 evaluaciones).
Cada corrida se guarda en `resultados/corridas/*.csv` junto con su `.json` (tiempo, configuración, contadores). El entorno (versiones, semillas) está en `resultados/entorno.json`.

**Demostración en vivo** (presupuesto 40, $N=10$, semilla 0; unos 15 s) y **comprobación de reproducibilidad**: NSGA-II con $N=20$ es secuencial, así que una corrida de presupuesto 60 debe coincidir exactamente con las primeras 60 evaluaciones de la corrida guardada de presupuesto 200.

In [8]:
# Demostración en vivo
cfg_demo = ConfigBusqueda(presupuesto=40, tam_poblacion=10)
for m in ("NSGA-II", "Aleatoria"):
    df, res = ejecutar_corrida(m, 0, cfg_demo)
    f = M.frente(df).sort_values("n_parametros")
    print(f"{m:9s}: {len(df)} evaluaciones reales, {res.tiempo_s:.1f} s, frente de {len(f)} arquitecturas, mejor error {df.val_error.min():.4f}")
display(f[["capas", "activacion", "n_parametros", "val_error"]].reset_index(drop=True).head(6))

NSGA-II  : 40 evaluaciones reales, 7.1 s, frente de 10 arquitecturas, mejor error 0.0279


Aleatoria: 40 evaluaciones reales, 7.3 s, frente de 10 arquitecturas, mejor error 0.0279


,capas,activacion,n_parametros,val_error
0,2-6-3,tanh,209,0.378830
1,3,relu,235,0.247911
2,4,relu,310,0.158774
3,5,relu,385,0.122563
4,8,tanh,610,0.075209
5,9-11,relu,815,0.066852


In [9]:
# Reproducibilidad: prefijo de 60 evaluaciones de NSGA-II (N=20) frente a la corrida guardada
guardada = pd.read_csv(DIR_RES / "corridas" / "principal__NSGA-II__s0.csv").sort_values("orden").head(60).reset_index(drop=True)
nueva, _ = ejecutar_corrida("NSGA-II", 0, ConfigBusqueda(presupuesto=60, tam_poblacion=20))
nueva = nueva.sort_values("orden").reset_index(drop=True)
iguales = (guardada[["capas", "activacion"]].equals(nueva[["capas", "activacion"]])
           and np.allclose(guardada.val_error, nueva.val_error, atol=0) and (guardada.n_parametros == nueva.n_parametros).all())
print("Reproducción idéntica de las primeras 60 evaluaciones:", iguales)
assert iguales

Reproducción idéntica de las primeras 60 evaluaciones: True


## 17. Resultados
Se leen todas las corridas guardadas. Primero, **comprobaciones de integridad** del diseño experimental; solo después se interpretan resultados.

In [10]:
ev_all, ej_all = cargar_resultados(DIR_RES)
ev, ej = ev_all[ev_all.tag == "principal"], ej_all[ej_all.tag == "principal"]

# Integridad: presupuesto, duplicados, finitud, equidad de datos
chequeos = []
for (tag, m, s), g in ev_all.groupby(["tag", "metodo", "semilla"]):
    esperado = {"principal": 200, "sens_pob10": 200, "sens_pob50": 200}[tag]
    assert len(g) == esperado, (tag, m, s, len(g))
    assert g.duplicated(["capas", "activacion"]).sum() == 0, "arquitectura duplicada"
    assert np.isfinite(g[["val_error", "n_parametros"]].to_numpy()).all()
    assert g.n_parametros.min() >= pmin and g.n_parametros.max() <= pmax
print(f"{ev_all.groupby(['tag','metodo','semilla']).ngroups} corridas: todas con 200 evaluaciones reales, sin duplicados, sin valores no finitos, dentro de los límites.")
print("Hijos reemplazados por muestra aleatoria (fallback) en NSGA-II:", int(ej_all[ej_all.metodo == 'NSGA-II'].fallback_aleatorio.sum()))
pareadas = ev.pivot_table(index="semilla", columns="metodo", values="orden", aggfunc="count")
print("Evaluaciones por (semilla, método) del conjunto principal:\n", pareadas.T.to_string())
n_ini = ev[ev.orden <= 20].sort_values(["semilla", "orden"])
mismo = all(n_ini[(n_ini.semilla == s) & (n_ini.metodo == "NSGA-II")][["capas", "activacion"]].reset_index(drop=True)
            .equals(n_ini[(n_ini.semilla == s) & (n_ini.metodo == "Aleatoria")][["capas", "activacion"]].reset_index(drop=True)) for s in SEMILLAS)
print("Primeras 20 evaluaciones idénticas en ambos métodos (diseño pareado):", mismo)
print(f"Entrenamientos que NO convergieron en 100 épocas: aleatoria {1-ev[ev.metodo=='Aleatoria'].convergio.mean():.1%}, NSGA-II {1-ev[ev.metodo=='NSGA-II'].convergio.mean():.1%}")

40 corridas: todas con 200 evaluaciones reales, sin duplicados, sin valores no finitos, dentro de los límites.
Hijos reemplazados por muestra aleatoria (fallback) en NSGA-II: 5
Evaluaciones por (semilla, método) del conjunto principal:
 semilla      0    1    2    3    4    5    6    7    8    9
metodo                                                     
Aleatoria  200  200  200  200  200  200  200  200  200  200
NSGA-II    200  200  200  200  200  200  200  200  200  200
Primeras 20 evaluaciones idénticas en ambos métodos (diseño pareado): True
Entrenamientos que NO convergieron en 100 épocas: aleatoria 49.5%, NSGA-II 54.0%


In [11]:
tabla = M.tabla_metricas(ev_all, ej_all)
tabla.to_csv(DIR_RES / "metricas_por_ejecucion.csv", index=False)
COLS = ["hv", "tam_frente", "mejor_val_error", "min_param_tol", "tiempo_s"]
resumen = M.resumen_descriptivo(tabla, COLS)
resumen.to_csv(DIR_RES / "resumen_metricas.csv", index=False)
p = resumen[resumen.tag == "principal"].set_index("metodo")
filas = []
for c_, nombre in [("hv", "Hipervolumen normalizado (↑)"), ("tam_frente", "Tamaño del frente (↑)"),
                   ("mejor_val_error", "Mejor error de validación (↓)"), ("min_param_tol", "Parámetros mínimos con error ≤ mejor+1 pp (↓)"),
                   ("tiempo_s", "Tiempo de búsqueda, s (informativo)")]:
    for m in ("NSGA-II", "Aleatoria"):
        filas.append({"métrica": nombre, "método": m, "media": p.loc[m, f"{c_}_media"], "d.e.": p.loc[m, f"{c_}_de"],
                      "mediana": p.loc[m, f"{c_}_mediana"], "IQR": p.loc[m, f"{c_}_iqr"]})
display(pd.DataFrame(filas).round(4).set_index(["métrica", "método"]))

media      d.e.    mediana       IQR
métrica                                       método                                             
Hipervolumen normalizado (↑)                  NSGA-II       0.9158    0.0093     0.9198    0.0103
                                              Aleatoria     0.9121    0.0094     0.9111    0.0120
Tamaño del frente (↑)                         NSGA-II      13.5000    2.7588    14.0000    3.0000
                                              Aleatoria    11.9000    2.0248    12.0000    2.5000
Mejor error de validación (↓)                 NSGA-II       0.0192    0.0081     0.0181    0.0070
                                              Aleatoria     0.0181    0.0065     0.0167    0.0042
Parámetros mínimos con error ≤ mejor+1 pp (↓) NSGA-II    1295.5000  427.3922  1206.0000  850.5000
                                              Aleatoria  1360.9000  606.8427  1285.0000  693.7500
Tiempo de búsqueda, s (informativo)           NSGA-II      36.9550    1.9663    36.3166    1.7548
                                              Aleatoria    39.4499    1.9127    38.8425    1.9882

**Resultado (10 corridas pareadas, 200 evaluaciones cada una).** El hipervolumen normalizado de NSGA-II tiene media 0,9158 (d.e. 0,0093) y mediana 0,9198 (IQR 0,0103); el de la búsqueda aleatoria, media 0,9121 (d.e. 0,0094) y mediana 0,9111 (IQR 0,0120).
El frente de NSGA-II tiene en promedio 13,5 arquitecturas y el de la aleatoria 11,9. El **mejor error de validación** es prácticamente el mismo (mediana 0,0181 frente a 0,0167; 5 de 10 pares empatan) y la arquitectura más pequeña que queda a ≤ 1 punto porcentual de ese mejor error tiene una mediana de 1206 parámetros en NSGA-II y 1285 en la aleatoria.
El tiempo de búsqueda (≈ 37 s frente a ≈ 39 s por corrida) es solo informativo: se midió con la máquina en estados de carga distintos (hubo reinicios del entorno) y **no es una medida de energía**.

**Integridad.** Las 40 corridas respetan el presupuesto exacto, sin duplicados ni valores no finitos. Solo 5 hijos de NSGA-II (de 5200 generados en las 30 corridas de NSGA-II) tuvieron que reemplazarse por una muestra aleatoria única, un efecto despreciable.
Entre el 49,5 % (aleatoria) y el 54,0 % (NSGA-II) de los entrenamientos **no convergieron** en 100 épocas: el error mide el rendimiento con un presupuesto de entrenamiento fijo, no el de redes entrenadas hasta converger (limitación, sección 22).

## 18. Visualizaciones
Las figuras de una sola corrida muestran la **semilla 0** (la primera, elegida antes de ver resultados; no se seleccionó por ser favorable). Las figuras agregadas usan las 10 semillas.

In [12]:
G.fig_frente_un_metodo(ev, "principal", "NSGA-II", 0, DIR_FIG / "fig1_frente_nsga2.png"); plt.show()
G.fig_frente_un_metodo(ev, "principal", "Aleatoria", 0, DIR_FIG / "fig2_frente_aleatoria.png"); plt.show()
G.fig_frentes_conjuntos(ev, "principal", 0, DIR_FIG / "fig3_frentes_conjuntos.png"); plt.show()

**Qué muestran.** Cada punto es una arquitectura entrenada; en gris están todas las evaluadas y en color las **no dominadas** (nadie las supera a la vez en error y en parámetros). El eje horizontal está en escala logarítmica.
**Resultado (semilla 0, ilustrativo, no evidencia por sí solo).** El frente tiene forma de "codo": el error cae de ≈ 0,36 con muy pocos parámetros a ≈ 0,06 hacia ≈ 550 parámetros, y luego mejora lentamente hasta ≈ 0,02 cerca de 1300. NSGA-II obtuvo 15 arquitecturas no dominadas y la aleatoria 12.
Como las primeras 20 evaluaciones son comunes, ambos frentes **coinciden en gran parte**; las diferencias visibles aparecen sobre todo entre ≈ 600 y 1300 parámetros, donde NSGA-II añade varias arquitecturas con errores de ≈ 0,03–0,04.
**Dominancia.** Todo punto gris está dominado por al menos un punto del frente: en promedio solo ≈ 7 % (NSGA-II) y ≈ 6 % (aleatoria) de las 200 evaluaciones llegan a ser no dominadas, es decir, ≈ 93–94 % son soluciones claramente dominadas.

In [13]:
G.fig_error_vs_parametros(ev, "principal", DIR_FIG / "fig4_error_vs_parametros.png"); plt.show()

**Qué muestra.** Las 2000 arquitecturas evaluadas por cada método (10 corridas), con error de validación frente a parámetros.
**Resultado.** En ambos métodos el error baja al aumentar los parámetros y se aplana hacia ≈ 1000–2000. Pero **a igual número de parámetros el error varía enormemente** (alrededor de 1000 parámetros hay desde ≈ 0,02 hasta ≈ 0,6): el número de parámetros no determina el desempeño; también importan la forma de la red (por ejemplo, cuellos de botella de pocas neuronas) y la activación.
La nube de NSGA-II es visiblemente más densa por debajo de ≈ 1000 parámetros. Descriptivamente, el **49,6 %** de sus evaluaciones usa menos de 1000 parámetros frente al **39,2 %** de la aleatoria (mediana de parámetros evaluados: 1020 frente a 1360).
**Interpretación (hipótesis, no probada):** NSGA-II reparte el presupuesto más hacia la zona de redes pequeñas, la que más pesa en el hipervolumen logarítmico.

In [14]:
G.fig_distribucion(tabla, "principal", "hv", "Hipervolumen normalizado por corrida (líneas = misma semilla)",
                   "Hipervolumen normalizado  ↑ mejor", DIR_FIG / "fig6_distribucion_hv.png"); plt.show()
curva = M.curva_hv(ev_all, [20, 40, 60, 100, 140, 200]); curva.to_csv(DIR_RES / "curva_hv.csv", index=False)
G.fig_convergencia(curva, "principal", DIR_FIG / "fig7_convergencia.png"); plt.show()

**Qué muestran.** *Distribución:* hipervolumen por corrida; cada línea gris une los dos métodos con la misma semilla (misma partición de datos). *Convergencia:* hipervolumen medio (± 1 d.e.) del archivo según las evaluaciones consumidas.
**Resultado.** En 8 de 10 semillas NSGA-II supera a la aleatoria, pero con diferencias pequeñas (mediana de la diferencia +0,0024) frente a la variabilidad entre semillas (d.e. ≈ 0,009). Por eso el análisis **pareado** es esencial: la partición de datos influye más que el método.
En la convergencia, ambas curvas empiezan iguales (por construcción, 20 evaluaciones comunes), son prácticamente iguales a 40 evaluaciones, NSGA-II va por delante a 60, 140 y 200, y a 100 están prácticamente empatadas. Las bandas de ± 1 d.e. se solapan en todo el rango.
**Interpretación:** la ventaja de NSGA-II, si existe, es pequeña e intermitente en este experimento; la gráfica no sustenta una superioridad clara.

## 19. Análisis estadístico
**Hipervolumen.** Área del espacio (error, log₁₀ parámetros) dominada por el frente y acotada por el punto de referencia **común** $(0{,}9;\ P_{\max})$, dividida por el área del rectángulo ideal–referencia, de modo que está en $[0,1]$. El error de referencia 0,9 es el de adivinar en 10 clases balanceadas;
$P_{\max}$ es la red mayor del espacio. El logaritmo refleja que pasar de 200 a 400 parámetros pesa más que de 12 000 a 12 200. Se calcula sobre el **archivo** (todo lo evaluado) de cada método, igual para ambos.
**Robustez a la elección de la métrica:** también se reporta con referencia de error 0,5 y con parámetros en escala lineal.

**Prueba.** Wilcoxon de rangos con signo (exacto, bilateral, por permutación; implementación propia validada contra scipy en ausencia de empates, que no depende de la versión de scipy: descarta diferencias nulas y usa rangos medios si hay empates) sobre los 10 pares con igual semilla; se informan victorias por método y la mediana de las diferencias. Con $n=10$ la potencia es baja: un $p$ alto **no demuestra equivalencia**.
Se usa $\alpha=0{,}05$ fijado antes de ver los resultados; las seis comparaciones siguientes son exploratorias y no se corrigen por múltiples pruebas (hacerlo solo reduciría aún más las diferencias significativas).

In [15]:
mets = [("hv", True), ("hv_ref050", True), ("hv_lineal", True), ("tam_frente", True), ("mejor_val_error", False), ("min_param_tol", False)]
comp = pd.DataFrame([M.comparacion_pareada(tabla, "principal", m, mayor) for m, mayor in mets])
comp.to_csv(DIR_RES / "comparaciones_wilcoxon.csv", index=False)
display(comp.round(4).set_index("metrica"))
cob = M.cobertura_entre_metodos(ev, "principal"); cob.to_csv(DIR_RES / "cobertura_C.csv", index=False)
print("Cobertura C de Zitzler (fracción del frente del otro método que es dominada o igualada; descriptiva):")
display(cob.drop(columns="semilla").agg(["mean", "median"]).round(3))

,n_pares,mediana_NSGA-II,mediana_Aleatoria,mediana_diferencia,victorias_NSGA2,victorias_Aleatoria,empates,p_wilcoxon_bilateral
metrica,,,,,,,,
hv,10,0.9198,0.9111,0.0024,8,2,0,0.1055
hv_ref050,10,0.8557,0.8421,0.0043,8,2,0,0.1055
hv_lineal,10,0.9723,0.9742,-0.0001,5,5,0,0.4922
tam_frente,10,14.0000,12.0000,1.5000,8,2,0,0.1836
mejor_val_error,10,0.0181,0.0167,0.0000,1,4,5,0.5625
min_param_tol,10,1206.0000,1285.0000,-64.5000,6,1,3,0.2656


Cobertura C de Zitzler (fracción del frente del otro método que es dominada o igualada; descriptiva):


,"C(NSGA-II,Aleatoria)","C(Aleatoria,NSGA-II)"
mean,0.776,0.624
median,0.764,0.600


**Resultado.** En ninguna de las seis comparaciones se rechaza $H_0$ con $\alpha=0{,}05$:
* *Hipervolumen (referencia 0,9, escala log):* NSGA-II gana en 8 de 10 pares, diferencia mediana +0,0024, $p=0{,}106$. Con referencia de error 0,5 el patrón es el mismo ($p=0{,}106$; son los mismos pares, no una evidencia independiente).
* *Hipervolumen con parámetros en escala lineal:* 5 victorias para cada método, diferencia mediana −0,0001, $p=0{,}49$. **La conclusión depende de cómo se valoran las redes pequeñas**: con escala lineal desaparece la ventaja.
* *Tamaño del frente:* 14 frente a 12 (medianas), 8–2, $p=0{,}18$. *Mejor error de validación:* sin diferencia ($p=0{,}56$, 5 empates). *Parámetros mínimos a ≤ +1 pp:* 1206 frente a 1285, $p=0{,}27$.
* *Cobertura de Zitzler (descriptiva):* en promedio, el 77,6 % del frente de la aleatoria queda dominado o igualado por el de NSGA-II, y el 62,4 % en sentido contrario; ambos valores son altos, en parte porque comparten las 20 primeras evaluaciones (explicación plausible, no verificada por separado).

**Interpretación.** Hay una **tendencia favorable a NSGA-II** en el hipervolumen logarítmico, pero con 10 pares la potencia es baja y no podemos afirmar ni superioridad ni equivalencia: un $p$ alto no demuestra que los métodos sean iguales. La hipótesis H1 **no queda confirmada** con este protocolo.

### Evaluación final de las soluciones seleccionadas (única vez que se usa el conjunto de prueba)
Para cada corrida se extrae el frente de **validación** y se eligen tres soluciones con criterios fijados antes (menor error, codo, compacta ≤ +1 pp). Solo entonces se reentrena la arquitectura con la misma semilla (el código verifica que reproduce exactamente el error de validación guardado) y se mide en **prueba**.
Como referencia de contexto, se evalúan tres MLP manuales con el mismo procedimiento.

In [16]:
sel = M.seleccion_final(ev); sel.to_csv(DIR_RES / "seleccion_final.csv", index=False)
orden_c = ["mejor_error", "codo", "compacta"]
res_sel = (sel.groupby(["criterio", "metodo"])[["n_parametros", "val_error", "test_error", "brecha_test_menos_val"]]
           .agg(["median", "mean", "std"]).round(4).loc[orden_c])
display(res_sel)
rows = []
for crit in orden_c:
    for met, mayor in (("test_error", False), ("n_parametros", False)):
        r_ = M.comparacion_pareada(sel[sel.criterio == crit], "principal", met, mayor); r_["criterio"] = crit; rows.append(r_)
comp_sel = pd.DataFrame(rows)[["criterio", "metrica", "mediana_NSGA-II", "mediana_Aleatoria", "victorias_NSGA2", "victorias_Aleatoria", "empates", "p_wilcoxon_bilateral"]]
comp_sel.to_csv(DIR_RES / "comparaciones_seleccion_final.csv", index=False); display(comp_sel.round(4))
ref = pd.read_csv(DIR_RES / "referencias_manuales.csv")
display(ref.groupby("referencia")[["n_parametros", "val_error", "test_error"]].agg(["median"]).round(4))
G.fig_mejores_soluciones(sel, "principal", DIR_FIG / "fig5_mejores_soluciones.png"); plt.show()

n_parametros                    val_error                 test_error                 brecha_test_menos_val                
                            median    mean        std    median    mean     std     median    mean     std                median    mean     std
criterio    metodo                                                                                                                              
mejor_error Aleatoria       2410.0  2859.8  1653.2044    0.0167  0.0181  0.0065     0.0292  0.0294  0.0074                0.0125  0.0113  0.0115
            NSGA-II         2372.5  2455.0  1084.0895    0.0181  0.0192  0.0081     0.0292  0.0300  0.0070                0.0152  0.0108  0.0099
codo        Aleatoria        460.0   475.0   131.3393    0.0641  0.0682  0.0194     0.0903  0.0856  0.0324                0.0193  0.0173  0.0242
            NSGA-II          497.5   482.5    86.9626    0.0627  0.0663  0.0268     0.0736  0.0803  0.0233                0.0110  0.0140  0.0193
compacta    Aleatoria       1285.0  1360.9   606.8427    0.0251  0.0245  0.0082     0.0375  0.0364  0.0073                0.0152  0.0119  0.0127
            NSGA-II         1206.0  1295.5   427.3922    0.0251  0.0253  0.0084     0.0375  0.0397  0.0134                0.0180  0.0144  0.0173

,criterio,metrica,mediana_NSGA-II,mediana_Aleatoria,victorias_NSGA2,victorias_Aleatoria,empates,p_wilcoxon_bilateral
0,mejor_error,test_error,0.0292,0.0292,2,3,5,0.8750
1,mejor_error,n_parametros,2372.5000,2410.0000,4,2,4,0.5625
2,codo,test_error,0.0736,0.0903,3,2,5,0.8125
3,codo,n_parametros,497.5000,460.0000,2,3,5,1.0000
4,compacta,test_error,0.0375,0.0375,2,3,5,0.3125
5,compacta,n_parametros,1206.0000,1285.0000,6,1,3,0.2656


,n_parametros,val_error,test_error
,median,median,median
referencia,,,
manual_grande,13130.0,0.0306,0.0306
manual_mediana,4810.0,0.0265,0.0236
manual_pequena,1210.0,0.0334,0.0333


**Resultado.** Con las soluciones elegidas en validación y medidas una sola vez en prueba (mediana de 10 corridas):
* *Menor error de validación:* ≈ 2372 parámetros (NSGA-II) y ≈ 2410 (aleatoria); error de prueba 0,0292 en ambos ($p=0{,}88$).
* *Compacta (≤ +1 pp):* ≈ 1206 y ≈ 1285 parámetros; error de prueba 0,0375 en ambos ($p=0{,}31$).
* *Codo:* ≈ 498 y ≈ 460 parámetros; error de prueba 0,0736 (NSGA-II) y 0,0903 (aleatoria), pero con 5 empates, 3 victorias frente a 2 y $p=0{,}81$: no hay evidencia de diferencia, y los rangos intercuartílicos son amplios.
* *Optimismo de la validación:* el error de prueba supera al de validación en ≈ 1,1–1,9 puntos porcentuales (mediana de la brecha) en todos los criterios, como cabe esperar porque la validación se usó para guiar la búsqueda y para elegir.

**Qué cuesta reducir el tamaño.** Pasar de la solución de menor error (≈ 2400 parámetros, ≈ 2,9 % de error de prueba) a la compacta (≈ 1200–1300 parámetros, la mitad) cuesta ≈ 0,8 puntos porcentuales; llegar al codo (≈ 500 parámetros, −79 %) cuesta ≈ 4,4 puntos (NSGA-II) a ≈ 6,1 (aleatoria).

**Referencias manuales (mediana de 10 particiones; descriptivas, sin prueba estadística).** La red mayor del espacio (64-64-64, 13 130 parámetros) obtuvo 0,0306 de error de prueba: **no mejora** a la solución de menor error encontrada por la búsqueda (0,0292 con unas 5,5 veces menos parámetros), lo que es coherente con que no hace falta la red más grande.
Pero **la búsqueda tampoco superó con claridad a conjeturas razonables**: una red manual de 64 neuronas (4810 parámetros) llegó a 0,0236, y una de 16 neuronas (1210 parámetros, tamaño similar al de la solución compacta) a 0,0333, frente a 0,0375 de la compacta. Posible explicación (hipótesis): la elección sobre una validación pequeña (359 muestras) sobrestima la calidad de lo elegido.

## 20. Análisis de sensibilidad (acotado)
Se varía **un factor a la vez**, manteniendo las demás condiciones constantes:
1. **Tamaño de población** de NSGA-II con presupuesto fijo de 200: $N\in\{10,20,50\}$ (19, 9 y 3 generaciones). La búsqueda aleatoria no depende de $N$.
2. **Presupuesto** $B\in\{20,40,60,100,140,200\}$ evaluaciones. Como NSGA-II ($N=20$) y la aleatoria son secuenciales con la misma semilla, el resultado con presupuesto $B$ es exactamente el prefijo de $B$ evaluaciones de la corrida de 200 (comprobado en la sección 16), así que no requiere reentrenar.
Las mismas 10 semillas en todos los casos.

In [17]:
mapa = {"sens_pob10": 10, "principal": 20, "sens_pob50": 50}
sp = M.resumen_descriptivo(tabla[tabla.metodo == "NSGA-II"], ["hv", "tam_frente", "min_param_tol"])
sp["N"] = sp.tag.map(mapa); sp = sp.sort_values("N")
alea = tabla[(tabla.tag == "principal") & (tabla.metodo == "Aleatoria")]
print(f"Referencia: búsqueda aleatoria hv mediana {alea.hv.median():.4f} (media {alea.hv.mean():.4f}, d.e. {alea.hv.std():.4f})")
display(sp[["N", "hv_media", "hv_de", "hv_mediana", "hv_iqr", "tam_frente_mediana", "min_param_tol_mediana"]].round(4).set_index("N"))
# Cada N frente a la aleatoria (mismas semillas)
filas = []
for tag, n in mapa.items():
    t = pd.concat([tabla[(tabla.tag == tag) & (tabla.metodo == "NSGA-II")],
                   tabla[(tabla.tag == "principal") & (tabla.metodo == "Aleatoria")].assign(tag=tag)])
    r_ = M.comparacion_pareada(t, tag, "hv"); r_["N"] = n; filas.append(r_)
sens_pob = pd.DataFrame(filas).sort_values("N")[["N", "mediana_NSGA-II", "mediana_Aleatoria", "mediana_diferencia", "victorias_NSGA2", "victorias_Aleatoria", "p_wilcoxon_bilateral"]]
sens_pob.to_csv(DIR_RES / "sensibilidad_poblacion.csv", index=False); display(sens_pob.round(4).set_index("N"))
G.fig_sensibilidad(tabla, "hv", "Hipervolumen normalizado  ↑ mejor", DIR_FIG / "fig8_sensibilidad_poblacion.png"); plt.show()
sens_pres = M.comparacion_por_presupuesto(curva, "principal"); sens_pres.to_csv(DIR_RES / "sensibilidad_presupuesto.csv", index=False)
display(sens_pres.round(4).set_index("presupuesto"))

Referencia: búsqueda aleatoria hv mediana 0.9111 (media 0.9121, d.e. 0.0094)


,hv_media,hv_de,hv_mediana,hv_iqr,tam_frente_mediana,min_param_tol_mediana
N,,,,,,
10,0.9167,0.0100,0.9195,0.0142,13.5,1022.5
20,0.9158,0.0093,0.9198,0.0103,14.0,1206.0
50,0.9159,0.0088,0.9181,0.0123,13.0,1210.0


,mediana_NSGA-II,mediana_Aleatoria,mediana_diferencia,victorias_NSGA2,victorias_Aleatoria,p_wilcoxon_bilateral
N,,,,,,
10,0.9195,0.9111,0.0021,6,3,0.0742
20,0.9198,0.9111,0.0024,8,2,0.1055
50,0.9181,0.9111,0.0017,8,2,0.0488


,mediana_hv_NSGA-II,mediana_hv_Aleatoria,mediana_diferencia,victorias_NSGA2,victorias_Aleatoria,empates,p_wilcoxon
presupuesto,,,,,,,
20,0.8276,0.8276,0.0000,0,0,10,1.0000
40,0.8569,0.8581,-0.0032,5,5,0,0.9219
60,0.8943,0.8762,0.0139,8,2,0,0.1309
100,0.9092,0.9009,0.0003,5,5,0,0.7695
140,0.9161,0.9087,0.0016,6,4,0,0.2754
200,0.9198,0.9111,0.0024,8,2,0,0.1055


**Resultado — tamaño de población (presupuesto fijo 200).** La mediana del hipervolumen de NSGA-II es 0,9195 ($N=10$), 0,9198 ($N=20$) y 0,9181 ($N=50$), siempre por encima de la aleatoria (0,9111). Las diferencias pareadas son +0,0021 ($p=0{,}074$), +0,0024 ($p=0{,}106$) y +0,0017 ($p=0{,}0488$).
Solo $N=50$ queda por debajo de 0,05 y por muy poco; como se hicieron varias comparaciones exploratorias sin corregir, **no lo interpretamos como evidencia sólida**. Las medias casi no cambian con $N$ (0,9167; 0,9158; 0,9159): en este rango el tamaño de población no parece ser un factor decisivo.

**Resultado — presupuesto.** A 20 evaluaciones ambos son idénticos (por construcción). La diferencia mediana de hipervolumen es −0,0032 a 40, +0,0139 a 60 (8–2, $p=0{,}13$), +0,0003 a 100 (5–5), +0,0016 a 140 y +0,0024 a 200. Ninguna es significativa y el signo y la magnitud **no crecen de forma monótona** con el presupuesto.
**Interpretación:** la sensibilidad observada es pequeña frente a la variabilidad entre particiones; la ventaja de NSGA-II aparece como un patrón débil y no estable en este espacio y con estos presupuestos. Un presupuesto de 60 evaluaciones corresponde a solo 2 generaciones de NSGA-II, así que esa cifra no debe leerse como convergencia.

In [18]:
# Exportación del archivo y de los frentes de cada corrida
ev_all.to_csv(DIR_RES / "evaluaciones_todas.csv", index=False)
frentes = pd.concat([M.frente(g).assign(tag=t, metodo=m, semilla=s) for (t, m, s), g in ev_all.groupby(["tag", "metodo", "semilla"])])
frentes.sort_values(["tag", "metodo", "semilla", "n_parametros"]).to_csv(DIR_RES / "frentes_pareto.csv", index=False)
# mediciones.csv con el esquema de la guía del docente (metrica_principal = hipervolumen normalizado, mismo significado siempre)
campos = ["ejecucion", "metodo", "instancia", "semilla", "metrica_principal", "tiempo_s", "evaluaciones", "observaciones"]
med = []
for i, (_, r) in enumerate(tabla.sort_values(["tag", "metodo", "semilla"]).iterrows(), 1):
    assert np.isfinite(r.hv), "ejecución con métrica no finita: registrarla aparte"
    med.append({"ejecucion": i, "metodo": r.metodo, "instancia": "digits" if r.tag == "principal" else f"digits_{r.tag}",
                "semilla": int(r.semilla), "metrica_principal": r.hv, "tiempo_s": round(r.tiempo_s, 2),
                "evaluaciones": int(r.evaluaciones),
                "observaciones": f"HV normalizado (ref error 0.9, P_max, log10 parametros); {int(r.n_no_convergio)} entrenamientos sin converger en 100 epocas"})
pd.DataFrame(med, columns=campos).to_csv(DIR_RES / "mediciones.csv", index=False)
print("Archivos en resultados/:", sorted(p.name for p in DIR_RES.glob("*.csv")))
print("Figuras en figuras/:", sorted(p.name for p in DIR_FIG.glob("*.png")))

Archivos en resultados/: ['cobertura_C.csv', 'comparaciones_seleccion_final.csv', 'comparaciones_wilcoxon.csv', 'curva_hv.csv', 'evaluaciones_todas.csv', 'frentes_pareto.csv', 'mediciones.csv', 'metricas_por_ejecucion.csv', 'referencias_manuales.csv', 'resumen_metricas.csv', 'seleccion_final.csv', 'sensibilidad_poblacion.csv', 'sensibilidad_presupuesto.csv']
Figuras en figuras/: ['fig1_frente_nsga2.png', 'fig2_frente_aleatoria.png', 'fig3_frentes_conjuntos.png', 'fig4_error_vs_parametros.png', 'fig5_mejores_soluciones.png', 'fig6_distribucion_hv.png', 'fig7_convergencia.png', 'fig8_sensibilidad_poblacion.png']


## 21. Discusión
Separamos **resultados** (medidos), **interpretaciones** (nuestra lectura) e **hipótesis** (no comprobadas).

**¿NSGA-II encontró mejores compromisos que la búsqueda aleatoria?**
*Resultado:* el hipervolumen logarítmico fue mayor en 8 de 10 pares (+0,0024 de mediana, $p=0{,}106$), el frente algo más grande (13,5 frente a 11,9) y la cobertura mayor; con parámetros en escala lineal no hubo ventaja (5–5), y a lo largo del presupuesto la ventaja fue intermitente.
*Interpretación:* hay una tendencia favorable pero **no concluyente**; no corresponde afirmar superioridad. *Hipótesis:* la tendencia proviene de que NSGA-II invierte más evaluaciones en redes pequeñas (49,6 % bajo 1000 parámetros frente a 39,2 %); no se comprobó causalmente.

**¿Qué arquitecturas ofrecen menor error?** Las de ≈ 2400 parámetros (mediana) tienen el menor error de validación (≈ 1,8 %) y ≈ 2,9 % en prueba; ambos métodos encuentran errores equivalentes (5 empates en 10 pares, $p=0{,}56$).
**¿Cuáles tienen menos parámetros?** Los extremos del frente se acercan al mínimo del espacio (160 parámetros), pero con errores de validación muy altos (p. ej. ≈ 0,36 en la semilla 0).
**¿Qué sacrificio implica reducir el tamaño?** Ver sección 19: ≈ 0,8 puntos porcentuales al reducir a la mitad los parámetros, y > 4 puntos al reducirlos ≈ 79 %. El compromiso es real y **no lineal**: el primer recorte es barato; el segundo, caro.
**¿Hay soluciones claramente dominadas?** Sí: ≈ 93–94 % de las evaluaciones de cada corrida.

**¿Mejora NSGA-II lo suficiente para justificar su costo?** El costo de cómputo es el mismo por diseño (200 entrenamientos) y el tiempo de búsqueda fue similar (≈ 37 s frente a ≈ 39 s, dominado por el entrenamiento y con las salvedades de la sección 22). El costo adicional es de **implementación y explicación**: más código, cinco hiperparámetros y operadores que justificar. Con la evidencia obtenida (ventaja pequeña, no significativa y sensible a la escala) **no podemos afirmar que se justifique**; la búsqueda aleatoria resultó una referencia exigente, como anticipa Bergstra y Bengio (2012).

**¿Qué falló y qué aprendimos?** (1) Entre el 50 % y el 54 % de los entrenamientos no convergieron: el error depende del presupuesto de entrenamiento fijado. (2) La validación guía y elige a la vez, y la prueba final reveló un optimismo de ≈ 1–2 pp. (3) El análisis de robustez mostró que la ventaja depende de la escala del hipervolumen: la métrica debe fijarse y justificarse **antes** de ver resultados. (4) Una primera versión de la Figura 5 usaba barras sobre escala logarítmica, que engañan por depender de un origen arbitrario; se rehízo con puntos y barras de error. (5) La ejecución se interrumpió dos veces: se diseñó el protocolo para ser reanudable y determinista.

**¿En qué condiciones podría cambiar la conclusión?** Con más semillas la tendencia podría volverse significativa o diluirse; con parámetros en escala lineal o con otra referencia de hipervolumen no hay ventaja visible; con más épocas de entrenamiento, otro conjunto de datos, otro presupuesto u operadores mejor ajustados el orden entre métodos podría cambiar. No lo evaluamos.

**¿Cómo elegir para un dispositivo limitado?** El frente se usa como un menú: (1) fijar el presupuesto de memoria (p. ej., a 4 bytes por parámetro en coma flotante de 32 bits, 1250 parámetros ocupan ≈ 5 KB: es aritmética, no una medición); (2) tomar del frente de validación la solución más precisa que quepa;
(3) verificar una sola vez en prueba; (4) **medir** latencia y energía en el dispositivo real, algo que aquí no se hizo y que los parámetros no sustituyen.
**¿Qué cambiaría con otro conjunto de datos?** *Hipótesis:* en problemas más fáciles o saturados (como mostró nuestro piloto con Breast Cancer) el frente sería corto y NSGA-II aportaría poco; en problemas más difíciles el compromiso sería más marcado y las diferencias entre métodos podrían ampliarse o desaparecer. No lo evaluamos.

**Relación con la literatura.** López-González et al. (2025) también compararon NSGA-II con búsqueda aleatoria mediante hipervolumen, con otros objetivos y dominio; no comparamos cifras. Vo y Luong (2024) advierten de la pérdida de diversidad en NAS multiobjetivo; nuestro frente algo mayor es compatible con diversidad adecuada, pero no la medimos directamente.

## 22. Limitaciones
1. **Un solo conjunto de datos, fácil y de baja resolución** (Digits 8×8); solo MLP densas de ≤ 3 capas y ≤ 64 neuronas. No se generaliza a otras tareas o modelos.
2. **Muestra pequeña:** 10 pares. La potencia es baja; un $p$ alto no prueba equivalencia. Varias comparaciones exploratorias **sin corrección** por pruebas múltiples; el $p=0{,}0488$ ($N=50$) es marginal.
3. **El número de parámetros no es energía ni latencia.** No se midió consumo energético ni tiempo de inferencia; no afirmamos ahorro energético. El tiempo de búsqueda es aproximado (carga variable de la máquina, reinicios del entorno).
4. **Entrenamiento corto y ruidoso:** 100 épocas fijas; 49,5–54,0 % de los entrenamientos no convergieron. Una sola inicialización de pesos por corrida (la misma para todas las arquitecturas), sin promediar el ruido de entrenamiento; el orden de arquitecturas podría cambiar con más épocas.
5. **Validación pequeña y reutilizada** (359 muestras; cada muestra = 0,28 pp): sirve para guiar la búsqueda y para elegir; produce optimismo (brecha de ≈ 1–2 pp respecto a prueba) y errores discretos.
6. **La métrica principal depende de decisiones:** referencia (0,9; $P_{\max}$) y escala logarítmica de parámetros. Con escala lineal desaparece la ventaja de NSGA-II.
7. **Diseño pareado con población inicial compartida** y una única semilla que controla a la vez la partición, la inicialización y el algoritmo: no se separa la variabilidad de datos de la del algoritmo.
8. **NSGA-II sin ajuste fino:** operadores y tasas fijados a priori; solo se varió $N$; 9 generaciones son pocas. Un mejor ajuste o más presupuesto podría cambiar el resultado.
9. **Reproducibilidad entre entornos:** verificada en el mismo entorno (versiones en `resultados/entorno.json`); no se probó en Colab ni en otros sistemas, y diferencias de bibliotecas numéricas podrían alterar los decimales.
10. **Literatura:** de varias fuentes se leyeron solo el resumen y la introducción; no se reprodujeron sus resultados.

## 23. Conclusiones
Las conclusiones se limitan a lo que los experimentos respaldan:
1. **H1 no queda confirmada.** Con 200 evaluaciones y 10 pares, NSGA-II tuvo mayor hipervolumen logarítmico en 8 de 10 corridas (+0,0024 de mediana), pero $p=0{,}106$; no rechazamos $H_0$. Tampoco demostramos equivalencia.
2. **La ventaja no es robusta:** desaparece con parámetros en escala lineal, es intermitente según el presupuesto (−0,0032 a +0,0139) y, según el tamaño de población, solo $N=50$ cruzó $p<0{,}05$, por poco y sin corrección.
3. **Descriptivamente**, NSGA-II invierte más evaluaciones en redes pequeñas (49,6 % frente a 39,2 % bajo 1000 parámetros) y obtiene frentes algo mayores (13,5 frente a 11,9). Es una hipótesis explicativa compatible con los datos, no un hecho probado.
4. **El compromiso error–tamaño es real:** reducir a la mitad los parámetros (≈ 2400 → ≈ 1250) costó ≈ 0,8 pp de error de prueba; reducirlos ≈ 79 % costó > 4 pp. La red más grande (13 130 parámetros) no superó a la mejor solución encontrada (0,0306 frente a 0,0292), aunque la búsqueda tampoco superó con claridad a las redes manuales de 64 y 16 neuronas.
5. **Metodología:** ambos métodos usaron exactamente el mismo presupuesto, particiones y entrenamiento (verificado por código y pruebas); la prueba solo se usó al final.

**Lo que NO puede concluirse:** ni la superioridad ni la equivalencia de NSGA-II (muestra pequeña, p > 0,05); nada sobre consumo de energía, latencia o memoria real (solo se cuenta el número de parámetros); nada sobre otros conjuntos de datos, tipos de red o presupuestos; y nada causal sobre *por qué* NSGA-II tiende a explorar redes pequeñas (es una hipótesis).

## 24. Trabajo futuro
* Más semillas (≥ 30) y corrección por pruebas múltiples; separar la semilla de partición, de inicialización y del algoritmo.
* Más conjuntos de datos (incluyendo uno donde el compromiso sea más exigente) y más objetivos reales: latencia o FLOPs medidos y **energía medida** en un dispositivo.
* Entrenar hasta convergencia (parada temprana) y promediar varias inicializaciones por arquitectura; usar validación cruzada o un conjunto de selección separado.
* Líneas base adicionales (muestreo por hipercubo latino, evolución regularizada) y otros algoritmos (NSGA-III, MOEA/D); ajustar operadores y tasas.
* Explorar si la mayor densidad de NSGA-II en redes pequeñas explica la tendencia (hipótesis de la sección 21).

## 25. Referencias
Datos bibliográficos comprobados en línea el 15-nov-2026 salvo donde se indica.
1. Deb, K., Pratap, A., Agarwal, S., & Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm: NSGA-II. *IEEE Transactions on Evolutionary Computation, 6*(2), 182–197. https://doi.org/10.1109/4235.996017
2. Zitzler, E., & Thiele, L. (1999). Multiobjective evolutionary algorithms: a comparative case study and the strength Pareto approach. *IEEE Transactions on Evolutionary Computation, 3*(4), 257–271. https://doi.org/10.1109/4235.797969
3. Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *Journal of Machine Learning Research, 13*, 281–305. https://jmlr.org/papers/v13/bergstra12a.html
4. López-González, Rodríguez y Greiner (2025). A multi-objective evolutionary computation approach for improving neural network-based surrogate models in structural engineering. *Algorithms, 18*(12), 754. https://doi.org/10.3390/a18120754 
5. Vo y Luong (2024). Efficient multi-objective neural architecture search via Pareto dominance-based novelty search. *Proceedings of GECCO 2024*. arXiv:2407.20656; https://doi.org/10.1145/3638529.3654064 
6. Khaled, Kaymak y Genga (2026). Alternating bi-objective optimization for explainable neuro-fuzzy systems. *arXiv preprint* arXiv:2602.19253. https://arxiv.org/abs/2602.19253 
7. ACM SIGEVO. (2026). *GECCO 2026 — Neuroevolution track*. https://gecco-2026.sigevo.org/Track?itemId=59 
8. Alpaydin, E., & Kaynak, C. (1998). *Optical Recognition of Handwritten Digits* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C50P49 (licencia CC BY 4.0; ficha consultada en línea).
9. Pedregosa, F., et al. (2011). Scikit-learn: machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. 

## 26. Bitácora y contribuciones de ambos integrantes
**Registro técnico del experimento (hechos verificables):**

| Hecho | Detalle |
|---|---|
| Piloto de selección de datos | 40 arquitecturas aleatorias en Digits y en Breast Cancer; se eligió Digits |
| Protocolo ejecutado | 10 semillas × 2 métodos × 200 evaluaciones (principal) + NSGA-II con $N=10$ y $N=50$ |
| Interrupciones | La ejecución se interrumpió dos veces por reinicio del entorno y se **reanudó** sin pérdida (cada corrida se guarda al terminar y es determinista) |
| Verificaciones | 60 pruebas automáticas; reproducción exacta de 60 evaluaciones; comprobaciones de integridad (sección 17) |
